## Hybrid Search vs Semantic Search vs KeyWord Search

 Does hybrid search actually win everywhere, or does it just average out wins and losses from BM25 and semantic search?

 we will  test three retrievers on BEIR FiQA (financial QA dataset, ~57k docs, 648 dev queries)

In [1]:
!pip install beir sentence-transformers rank_bm25 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 12.9 MB/s eta 0:00:00


In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="dev")

print(f"Corpus size: {len(corpus)}")
print(f"Queries: {len(queries)}")
print(f"Qrels: {len(qrels)}")

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


datasets/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

Corpus size: 57638
Queries: 500
Qrels: 500


### Retriever 1: BM25

Pure lexical/sparse retrieval. Good baseline — strong on exact keyword matches, weak on paraphrases.

In [3]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [doc["text"].lower().split() for doc in corpus.values()]
corpus_ids = list(corpus.keys())
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [corpus_ids[i] for i in top_indices], scores[top_indices]

## Retriever 2: Semantic (Sentence Transformers)

Dense embedding retrieval using all-MiniLM-L6-v2. Should handle paraphrases and natural language queries better than BM25, but may miss exact technical terms it hasn't seen the right context for.

Encoding 57k documents takes a few minutes on Colab — this is a one-time cost, cached for the rest of the notebook.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = [corpus[cid]["text"] for cid in corpus_ids]
corpus_embeddings = model.encode(corpus_texts, batch_size=64, show_progress_bar=True)

def semantic_search(query, top_k=10):
    query_embedding = model.encode([query])
    scores = np.dot(corpus_embeddings, query_embedding.T).squeeze()
    top_indices = scores.argsort()[-top_k:][::-1]
    return [corpus_ids[i] for i in top_indices], scores[top_indices]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/901 [00:00<?, ?it/s]

## Retriever 3: Hybrid (Score Fusion)

Normalize both score sets to [0,1] and combine with a weighted sum (alpha=0.5, equal weight).
This is the simplest fusion method — no RRF, no learned weights — deliberately, so the comparison stays interpretable.

In [5]:
def hybrid_search(query, top_k=10, alpha=0.5):
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)

    query_emb = model.encode([query])
    sem_scores = np.dot(corpus_embeddings, query_emb.T).squeeze()
    sem_norm = (sem_scores - sem_scores.min()) / (sem_scores.max() - sem_scores.min() + 1e-9)

    hybrid_scores = alpha * bm25_norm + (1 - alpha) * sem_norm
    top_indices = hybrid_scores.argsort()[-top_k:][::-1]
    return [corpus_ids[i] for i in top_indices], hybrid_scores[top_indices]

In [6]:
def recall_at_k(retrieved_ids, relevant_ids, k=10):
    retrieved_k = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    return len(retrieved_k & relevant) / len(relevant)

def mrr(retrieved_ids, relevant_ids):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

## Comparison
Baseline numbers before breaking down by category.

In [ ]:
def evaluate_retriever(search_fn, queries, qrels, k=10):
    recalls, mrrs = [], []
    for qid, query_text in queries.items():
        if qid not in qrels:
            continue
        relevant_ids = list(qrels[qid].keys())
        retrieved_ids, _ = search_fn(query_text, top_k=k)
        recalls.append(recall_at_k(retrieved_ids, relevant_ids, k))
        mrrs.append(mrr(retrieved_ids, relevant_ids))
    return np.mean(recalls), np.mean(mrrs)

bm25_recall, bm25_mrr = evaluate_retriever(bm25_search, queries, qrels)
sem_recall, sem_mrr = evaluate_retriever(semantic_search, queries, qrels)
hybrid_recall, hybrid_mrr = evaluate_retriever(hybrid_search, queries, qrels)

print(f"BM25:     Recall@10={bm25_recall:.3f}  MRR={bm25_mrr:.3f}")
print(f"Semantic: Recall@10={sem_recall:.3f}  MRR={sem_mrr:.3f}")
print(f"Hybrid:   Recall@10={hybrid_recall:.3f}  MRR={hybrid_mrr:.3f}")

BM25:     Recall@10=0.227  MRR=0.240
Semantic: Recall@10=0.467  MRR=0.453
Hybrid:   Recall@10=0.396  MRR=0.384


## Conclusion

Semantic search won outright on FiQA — Recall@10 of 0.467 and MRR of 0.453, well ahead of BM25 (0.227 / 0.240). Hybrid landed in between (0.396 / 0.384), beating neither.

This isn't the result most RAG content implies. "Hybrid search is better" is usually stated as a given. Here, a naive 50/50 score fusion didn't add BM25 and semantic together — it diluted the stronger retriever with the weaker one. Equal-weight fusion assumes both retrievers are contributing comparable signal. On this dataset, they aren't.

Two takeaways I'm carrying into later days:

1. **Fusion weight matters more than fusion itself.** Alpha=0.5 was never tuned — it was a default. Hybrid's actual ceiling is probably higher than what this run shows, and the honest fix is to sweep alpha rather than assume the midpoint is safe.
2. **"Hybrid is better" is dataset-dependent, not a law.** FiQA queries leaned semantic/paraphrase-heavy over exact-keyword. A dataset with more exact technical lookups might flip this entirely — which is exactly why category-level breakdowns matter more than one aggregate number.

Not tuning alpha before publishing this is the honest gap in today's build. Next: Day 25 starts the CoT reasoning order permutation experiment.